# Phase 4b — fine-tuning (Arms C and D)

Thin Colab runner: Unsloth **bf16 LoRA** on `Qwen/Qwen3.5-4B` (r=16, alpha=16, dropout 0, lr 2e-4 cosine, ~3% warmup, 2 epochs, max_seq_length 2048, effective batch 16, seed 0) on training data v1, loss on the completion only, same prompt as Arm A/C. The adapter is saved to Google Drive. Then:

- **Arm C**: the adapter, no context;
- **Arm D**: the adapter + the frozen B2 retrieval config, with the **Arm C output as the draft** (the fine-tuned model drafts for itself).

**Runtime:** GPU, **L4 (24 GB)**, the same GPU as Arms A/B/B2; the first cell stops on any other GPU (`EXPECTED_GPU`).

**`SKIP_TRAINING`** (default `False`): when `True`, step 1 does not train; it loads the existing adapter from `$DRIVE_DIR/adapters/qwen35-4b-polars-lora-v1` (and stops if it is missing) and keeps the training run's `train_meta.json` (`results/C_a100/train_meta.json`, the A100 training run). This is the setting for the L4 hardware-parity rerun of C/D with the same adapter (PLAN.md, "Hardware parity").

In [ ]:
# Run settings + GPU check.
SKIP_TRAINING = False   # True: no training; reuse the adapter already on Drive (L4 rerun of C/D)
EXPECTED_GPU = 'L4'     # official C/D runs use the L4, like A/B/B2; None disables the check
import torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('GPU:', GPU_NAME, '| SKIP_TRAINING =', SKIP_TRAINING)
if EXPECTED_GPU and (GPU_NAME is None or EXPECTED_GPU not in GPU_NAME):
    raise RuntimeError(f'expected an {EXPECTED_GPU} GPU, got {GPU_NAME!r}: change the Colab runtime type')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Google Drive: outputs are copied here so they survive the Colab session.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/polars-rag-vs-ft'
!mkdir -p $DRIVE_DIR

In [ ]:
# Clone the repo (idempotent) and enter it.
import os
REPO = 'https://github.com/yusuftanriverdi24/Polars-Rag---Finetune.git'
if not os.path.isdir('polars-rag-vs-ft'):
    !git clone -q $REPO polars-rag-vs-ft
%cd polars-rag-vs-ft
!git pull -q

In [ ]:
# Install pinned dependencies: `colab` = GPU stack, `rag` = sentence-transformers +
# LanceDB (exact pins). torch stays Colab's preinstalled CUDA build.
!pip install -q -e ".[colab,rag]"
# Deliberately NO fast linear-attention kernels here (causal-conv1d / flash-linear-attention, unlike
# notebook 05): Arms C and D must run the same Qwen3.5 code path (torch fallback) as Arms A/B/B2.

In [ ]:
# Training data: committed in the repo (data/sft/), else taken from notebook 05's zip on Drive.
import os
if not os.path.exists('data/sft/train.jsonl'):
    !unzip -o -q $DRIVE_DIR/results_datagen.zip
!wc -l data/sft/train.jsonl data/sft/dev.jsonl

In [ ]:
# 1. Fine-tune (wall time, peak VRAM and train/dev losses -> results/C/train_meta.json),
#    or with SKIP_TRAINING reuse the adapter on Drive and the training run's train_meta.json.
import os, shutil, subprocess
ADAPTER_DIR = DRIVE_DIR + '/adapters/qwen35-4b-polars-lora-v1'
os.makedirs('results/C', exist_ok=True)
if SKIP_TRAINING:
    missing = [f for f in ('adapter_config.json', 'adapter_model.safetensors')
               if not os.path.isfile(os.path.join(ADAPTER_DIR, f))]
    if missing:
        raise FileNotFoundError(
            f'SKIP_TRAINING=True but the adapter at {ADAPTER_DIR} is missing {missing}. '
            'Unzip qwen35-4b-polars-lora-v1.zip into $DRIVE_DIR/adapters/ or set SKIP_TRAINING=False.')
    shutil.copy('results/C_a100/train_meta.json', 'results/C/train_meta.json')
    print('Training skipped; using adapter', ADAPTER_DIR)
else:
    subprocess.run(['python', '-m', 'polars_llm.train.sft', '--output-dir', ADAPTER_DIR], check=True)
    subprocess.run('cd "$DRIVE_DIR/adapters" && zip -q -r qwen35-4b-polars-lora-v1.zip '
                   "qwen35-4b-polars-lora-v1 -x '*/checkpoints/*'",
                   shell=True, check=True, env={**os.environ, 'DRIVE_DIR': DRIVE_DIR})
!ls -la "$DRIVE_DIR/adapters" "$ADAPTER_DIR"

In [ ]:
# 2. Arm C: the adapter, no context.
!python scripts/generate.py --arm C --model Qwen/Qwen3.5-4B --adapter "$ADAPTER_DIR" --batch-size 8
!python scripts/score.py --arm C

In [ ]:
# 3. Arm D: frozen B2 retrieval config (corpus v2 + granite, draft-code query, k=8, 4,000 tokens) with the
#    Arm C output as the draft. Needs the v2 granite index.
!python -m polars_llm.rag.ingest
!python -m polars_llm.rag.chunk --stats /tmp/chunk_stats_v1.json
!python -m polars_llm.rag.corpus --stats /tmp/corpus_v2_stats.json
!python -m polars_llm.rag.index --chunks data/rag/chunks_v2.jsonl --index-dir data/rag/index_v2_granite --embedder granite --device cuda --batch-size 32
!python scripts/generate.py --arm D --model Qwen/Qwen3.5-4B --adapter "$ADAPTER_DIR" --batch-size 8 --retrieval-config results/rag/retrieval_eval_v2.json --drafts results/C/generations.jsonl
!python scripts/score.py --arm D

In [ ]:
# 4. Same determinism check every arm gets.
!python benchmark/determinism_audit.py --scores results/C/scores.json --out results/C/determinism_audit.json | tail -n 5
!python benchmark/determinism_audit.py --scores results/D/scores.json --out results/D/determinism_audit.json | tail -n 5

In [ ]:
# 5. Summary + zip (also copied to Drive).
!python -c "import json; [print(a, json.load(open(f'results/{a}/generation_meta.json'))['gpu'], json.load(open(f'results/{a}/scores.json'))['metrics']['pass_at_1'], json.load(open(f'results/{a}/scores.json'))['metrics']['status_counts']) for a in ('C', 'D')]"
!python -c "import json; m=json.load(open('results/C/train_meta.json')); print('train', m['wall_time_s'], 's, peak VRAM', m['peak_vram_gb'], 'GB, dev loss', m['dev_loss_by_epoch']); print('D cost', json.load(open('results/D/generation_meta.json'))['cost'])"
!zip -q -r results_CD.zip results/C results/D
!cp results_CD.zip $DRIVE_DIR/
from google.colab import files
files.download('results_CD.zip')